# cycle-detection-temp-set — faded example 1: Complete the gray-set back-edge check in a boolean detector

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `cycle-detection-temp-set`. The last cell reports your progress on the `Backprop: cycle detection via temp set` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: cycle detection via temp set` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`cycle-detection-temp-set`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "cycle-detection-temp-set"
DD_SUBTOPIC = "Backprop: cycle detection via temp set"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Boolean cycle detection keeps `perm` (finished) and `temp` (on the recursion stack) sets. The single decisive line is the gray check: re-entering a vertex that is still in `temp` is a back-edge and therefore a cycle. Forgetting this — or checking `perm` instead — turns the detector into a broken single-visited-set version that false-positives on diamonds.

## Faded exercise 1

Implement `has_cycle(adj, start)` over a `dict[str, list[str]]` adjacency map, returning `True` iff a cycle is reachable from `start`. The two-set skeleton and the un-graying logic are written for you. **Complete the gray-set back-edge test** that decides whether re-entering the current vertex means a cycle.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
def has_cycle(adj, start):
    perm, temp = set(), set()

    def visit(u):
        if u in perm:
            return False
        cycle_here = None  # TODO: fill in this step — read the prompt cell above
        if cycle_here:
            return True
        temp.add(u)
        for v in adj.get(u, []):
            if visit(v):
                return True
        temp.remove(u)
        perm.add(u)
        return False

    return visit(start)


def _test():
    dag = {"a": ["b", "c"], "b": ["d"], "c": ["d"], "d": []}
    cyc = {"a": ["b"], "b": ["c"], "c": ["a"]}
    self_loop = {"a": ["a"]}
    long_dag = {"n0": ["n1"], "n1": ["n2"], "n2": []}
    assert has_cycle(dag, "a") is False, "diamond DAG must not be a cycle"
    assert has_cycle(cyc, "a") is True, "a->b->c->a is a cycle"
    assert has_cycle(self_loop, "a") is True, "self-loop is a cycle"
    assert has_cycle(long_dag, "n0") is False, "chain is acyclic"


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def has_cycle(adj, start):
    perm, temp = set(), set()

    def visit(u):
        if u in perm:
            return False
        cycle_here = u in temp  # back-edge: u still on the DFS stack
        if cycle_here:
            return True
        temp.add(u)
        for v in adj.get(u, []):
            if visit(v):
                return True
        temp.remove(u)
        perm.add(u)
        return False

    return visit(start)
```
</details>